In [29]:
import pandas as pd
import numpy as np
import copy
from nltk.translate.bleu_score import sentence_bleu
import ray

# Suggestion
"""
in percentage
< 10 	Almost useless
10 - 19 	Hard to get the gist
20 - 29 	The gist is clear, but has significant grammatical errors
30 - 40 	Understandable to good translations
40 - 50 	High quality translations
50 - 60 	Very high quality, adequate, and fluent translations
> 60 	Quality often better than human
"""
# https://cloud.google.com/translate/automl/docs/evaluate

@ray.remote
def get_bleu_score(remaining_sentences, sentence):
    ans = sentence_bleu(remaining_sentences, sentence)
    return ans

def calculate_selfBleu(sentences):
    '''
    sentences - list of sentences generated by NLG system
    '''
    bleu_scores = []
    bleu_scores_refs = []

    for i in sentences:
        sentences_copy = copy.deepcopy(sentences)
        sentences_copy.remove(i)
        bleu_scores_refs.append(get_bleu_score.remote(sentences_copy, i))

    bleu_scores = ray.get(bleu_scores_refs)
    return np.mean(bleu_scores)

In [30]:
# init ray
ray.init(ignore_reinit_error=True)

df = pd.read_csv('../test/genai_questions_809156896092057937.csv')
# df = pd.read_csv('../test/genai_questions_311016922297059474.csv')
# print (df.head(10))

questions = df['questions'].to_list()
questions = list(set(questions))
sentences = [item.split() for item in questions]

ans = calculate_selfBleu(sentences)
print (ans)

2024-01-14 11:00:33,794	INFO worker.py:1558 -- Calling ray.init() again after it has already been called.


0.6157306021947423


In [32]:
ray.init(ignore_reinit_error=True)

df = pd.read_csv('./existing_qa_datasets/squad_v2_q.csv')
questions = df['question'].to_list()

ans = calculate_selfBleu(questions)
print (ans)

(raylet) [2024-01-14 11:11:39,903 C 19848 13052] (raylet.exe) dlmalloc.cc:129:  Check failed: *handle != nullptr CreateFileMapping() failed. GetLastError() = 1450
(raylet) *** StackTrace Information ***
(raylet) unknown
(raylet) recalloc
(raylet) BaseThreadInitThunk
(raylet) RtlUserThreadStart
(raylet) 


RaySystemError: System error: Unknown error

(raylet) The node with node id: 6b7a6375d2d6c443f46624968a701f15231829468a56be2a18ef1eac and address: 127.0.0.1 and node name: 127.0.0.1 has been marked dead because the detector has missed too many heartbeats from it. This can happen when a 	(1) raylet crashes unexpectedly (OOM, preempted node, etc.) 
	(2) raylet has lagging heartbeats due to slow network or busy workload.


In [33]:
ray.init(ignore_reinit_error=True)

df = pd.read_csv('./existing_qa_datasets/hotpot_q.csv')
questions = df['question'].to_list()

ans = calculate_selfBleu(questions)
print (ans)

2024-01-14 11:17:48,788	INFO worker.py:1558 -- Calling ray.init() again after it has already been called.


RaySystemError: System error: Unknown error